# Step 6 — Generate Embeddings

At this stage we will:

1. Read all data from the `chunks` table.
2. Create embeddings using the `BAAI/bge-small-en-v1.5` model.
3. Store embeddings to the `embeddings` table.

> Make sure the `sentence-transformers` package is installed.

In [8]:
from pathlib import Path
import sqlite3

DB_PATH = Path("../data/linux_docs.db")
assert DB_PATH.exists(), "Database tidak ditemukan."

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()


## Create `embeddings` table

In [9]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS embeddings (
    chunk_id INTEGER PRIMARY KEY,
    model TEXT NOT NULL,
    dimension INTEGER NOT NULL,
    embedding BLOB NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY(chunk_id) REFERENCES chunks(id)
);
''')

conn.commit()
print("✅ Embeddings table ready.")

✅ Embeddings table ready.


## Load embedding model

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: BAAI/bge-small-en-v1.5


## Read chunks

In [11]:
cursor.execute('''
SELECT id, content
FROM chunks
ORDER BY id;
''')

chunks = cursor.fetchall()
print(f"Total chunks: {len(chunks)}")

Total chunks: 41


## Generate embeddings and save

In [12]:
cursor.execute("DELETE FROM embeddings")

for row in chunks:
    vector = model.encode(
        row["content"],
        normalize_embeddings=True
    )

    vector = np.asarray(vector, dtype=np.float32)

    cursor.execute(
        '''
        INSERT INTO embeddings(
            chunk_id,
            model,
            dimension,
            embedding
        )
        VALUES (?, ?, ?, ?)
        ''',
        (
            row["id"],
            MODEL_NAME,
            vector.shape[0],
            vector.tobytes()
        )
    )

conn.commit()

print("✅ Embeddings successfully saved.")

✅ Embeddings successfully saved.


## Verification

In [13]:
cursor.execute('''
SELECT
    chunk_id,
    model,
    dimension,
    length(embedding) AS bytes
FROM embeddings
LIMIT 5;
''')

for row in cursor.fetchall():
    print(dict(row))


{'chunk_id': 83, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 84, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 85, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 86, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}
{'chunk_id': 87, 'model': 'BAAI/bge-small-en-v1.5', 'dimension': 384, 'bytes': 1536}


## Notes

Embeddings are stored as **float32 BLOB**.

Advantages of this approach:

- Database remains a single SQLite file.
- Easy to export to Qdrant, pgvector, or other systems in the future.
- In Step 7 we will read the BLOB again and perform **semantic search** using cosine similarity.

In [14]:
conn.close()
print("Done ✅")

Done ✅
